In [ ]:
import os
import re
import torch
import pandas as pd
from tqdm.notebook import tqdm 
from PIL import Image
from unsloth import FastVisionModel
from dotenv import load_dotenv

load_dotenv()

os.environ["WANDB_DISABLED"] = "true"

In [ ]:
LORA_DIR = os.getenv("LORA_DIR")
TEST_DIR = os.getenv("TEST_DIR")
TEST_CSV = os.getenv("TEST_CSV")
OUTPUT_RESULT_CSV = os.getenv("OUTPUT_RESULT_CSV")

# Jumlah gambar yang ingin dievaluasi untuk tes 
SAMPLE_SIZE = 1000

In [ ]:

print("\n[INFO] Membangunkan Qwen3-VL-2B dengan memori LoRA...")
model, tokenizer = FastVisionModel.from_pretrained(
    LORA_DIR,
    load_in_4bit=True,
)

FastVisionModel.for_inference(model)
print("[SELESAI] Model siap digunakan!")

In [ ]:

INSTRUCTION = "Analisis ekspresi wajah siswa ini secara detail. Tentukan tingkat kebosanan (boredom), ketertarikan (engagement), kebingungan (confusion), dan frustrasinya (frustration)."

def extract_scores(text):
    scores = {"Boredom": -1, "Engagement": -1, "Confusion": -1, "Frustration": -1}
    # Regex untuk menangkap angka 
    patterns = {
        "Boredom": r"(?:boredom|kebosanan)[\s\:\-\=]*(\d)",
        "Engagement": r"(?:engagement|ketertarikan)[\s\:\-\=]*(\d)",
        "Confusion": r"(?:confusion|kebingungan)[\s\:\-\=]*(\d)",
        "Frustration": r"(?:frustration|frustrasi)[\s\:\-\=]*(\d)"
    }
    
    for emotion, pattern in patterns.items():
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            scores[emotion] = int(match.group(1))
            
    return scores

In [ ]:

print(f"[INFO] Membaca data ground-truth dari: {TEST_CSV}")
df = pd.read_csv(TEST_CSV)

if len(df) > SAMPLE_SIZE:
    print(f"[INFO] Dataset terlalu besar ({len(df)} gambar). Mengambil sampel acak {SAMPLE_SIZE} gambar...")
    df = df.sample(n=SAMPLE_SIZE, random_state=42).reset_index(drop=True)

emotions = ["Boredom", "Engagement", "Confusion", "Frustration"]

exact_matches = {e: 0 for e in emotions}
adjacent_matches = {e: 0 for e in emotions}
valid_predictions = {e: 0 for e in emotions}

results_data = []
print(f"[SELESAI] Data siap diproses!")

In [ ]:

print(f"\n[INFO] Memulai evaluasi cepat ({SAMPLE_SIZE} gambar)...\n")

for index, row in tqdm(df.iterrows(), total=len(df), desc="Evaluasi Qwen"):
    img_name = str(row['Image_Name'])
    if not img_name.endswith(('.jpg', '.png')): 
        img_name += '.jpg'
        
    img_path = os.path.join(TEST_DIR, img_name)
    
    if not os.path.exists(img_path):
        continue
        
    try:
        image = Image.open(img_path).convert("RGB")
        messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": INSTRUCTION}]}]
        input_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
        inputs = tokenizer(image, input_text, add_special_tokens=False, return_tensors="pt").to("cuda")
        
        # max_new_tokens dinaikkan agar kalimat tidak terpotong
        outputs = model.generate(**inputs, max_new_tokens=500, use_cache=True, temperature=0.3)
        prompt_length = inputs["input_ids"].shape[1]
        response_text = tokenizer.decode(outputs[0][prompt_length:], skip_special_tokens=True)
        
        pred_scores = extract_scores(response_text)
        result_row = {"Image_Name": img_name, "Response_Text": response_text}
        
        for e in emotions:
            true_val = int(row[e])
            pred_val = pred_scores[e]
            
            result_row[f"True_{e}"] = true_val
            result_row[f"Pred_{e}"] = pred_val
            
            if pred_val != -1:
                valid_predictions[e] += 1
                if pred_val == true_val:
                    exact_matches[e] += 1
                if abs(pred_val - true_val) <= 1:
                    adjacent_matches[e] += 1
                    
        results_data.append(result_row)
        
    except Exception as e:
        pass

# SIMPAN & TAMPILKAN HASIL
results_df = pd.DataFrame(results_data)
results_df.to_csv(OUTPUT_RESULT_CSV, index=False)

print("\n" + "="*50)
print(f"🎯 LAPORAN AKURASI CEPAT QWEN3-VL-2B (SAMPEL {SAMPLE_SIZE} GAMBAR)")
print("="*50)

for e in emotions:
    if valid_predictions[e] > 0:
        exact_acc = (exact_matches[e] / valid_predictions[e]) * 100
        adj_acc = (adjacent_matches[e] / valid_predictions[e]) * 100
        print(f"[{e.upper()}] (Valid: {valid_predictions[e]} jawaban)")
        print(f"  ▶ Exact Accuracy (Tepat)      : {exact_acc:.2f}%")
        print(f"  ▶ Adjacent Accuracy (Toleransi): {adj_acc:.2f}%")
        print("-" * 50)
        
print(f"\n[INFO] Detail tebakan disimpan di: {OUTPUT_RESULT_CSV}")